# 22-29 · Flask поверх SQLite

Практика к разделу [«Добавляем SQLite в Flask-приложение»](../../site/chapters/glava-22/22-29-flask-sqlite.html). Полный проект — `projects/flask/todo-app/app.py`.

## Цель

Собрать маленькое Flask-приложение поверх настоящей базы SQLite и убедиться, что данные переживают создание нового объекта приложения — то есть ведут себя так же, как после перезапуска процесса.

## Рабочий пример

In [1]:
import sqlite3
import tempfile
import os

from flask import Flask, g, jsonify, redirect, request, url_for

put_k_baze = os.path.join(tempfile.mkdtemp(), "zadachi.db")


def sozdat_prilozhenie():
    prilozhenie = Flask(__name__)
    prilozhenie.config["DATABASE"] = put_k_baze

    def get_db():
        if "db" not in g:
            g.db = sqlite3.connect(prilozhenie.config["DATABASE"])
            g.db.row_factory = sqlite3.Row
        return g.db

    @prilozhenie.teardown_appcontext
    def close_db(exception=None):
        db = g.pop("db", None)
        if db is not None:
            db.close()

    @prilozhenie.route("/api/tasks", methods=["GET", "POST"])
    def api_tasks():
        db = get_db()
        if request.method == "POST":
            db.execute("INSERT INTO tasks (title) VALUES (?)", (request.get_json()["title"],))
            db.commit()
            return "", 201
        stroki = db.execute("SELECT id, title, done FROM tasks ORDER BY id").fetchall()
        return jsonify([dict(s) for s in stroki])

    with prilozhenie.app_context():
        get_db().execute("CREATE TABLE IF NOT EXISTS tasks (id INTEGER PRIMARY KEY, title TEXT NOT NULL, done INTEGER NOT NULL DEFAULT 0)")
        get_db().commit()

    return prilozhenie


prilozhenie1 = sozdat_prilozhenie()
client1 = prilozhenie1.test_client()
client1.post("/api/tasks", json={"title": "Задача до перезапуска"})

otvet1 = client1.get("/api/tasks")
print("До «перезапуска»:", otvet1.get_json())

До «перезапуска»: [{'done': 0, 'id': 1, 'title': 'Задача до перезапуска'}]


## Проверка результата — новый объект приложения видит те же данные

In [2]:
prilozhenie2 = sozdat_prilozhenie()   # как будто процесс перезапустили
client2 = prilozhenie2.test_client()

otvet2 = client2.get("/api/tasks")
print("После «перезапуска»:", otvet2.get_json())

assert otvet2.get_json() == otvet1.get_json()
assert len(otvet2.get_json()) == 1
print("Верно: данные хранятся в файле базы данных, а не в памяти процесса — новый объект приложения видит их.")

После «перезапуска»: [{'done': 0, 'id': 1, 'title': 'Задача до перезапуска'}]
Верно: данные хранятся в файле базы данных, а не в памяти процесса — новый объект приложения видит их.
